> ## SOLUTION / ANSWER KEY &mdash; Lab 7.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-observability.ipynb`](../lab-10-observability.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.10 &mdash; Observability: Log Every Run

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Record every stage of a run in an auditable log
- Read back the ordered trace for debugging
- Measure quality over a batch (the approval rate)

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Failure modes & observability](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-07-10")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
You can't run unattended what you can't see (deck slide 18). **Observability** means logging every
run &mdash; the **trigger**, each **tool call &amp; observation**, the **draft**, the **validation**
result, and the **human decision**. That log lets you **debug** a bad output, **audit** what
happened, and **measure** quality over time (approval rate, edit rate). Real stacks &mdash;
**LangSmith / Langfuse** &mdash; capture exactly this via LangChain callbacks; here you build the
minimal offline version, and it drops straight onto the real agent trace from Lab 11.

In [ ]:
# One run produces a sequence of stage events; a batch of runs produces a metric.
print("we log: trigger -> gather -> draft -> validate -> approve, plus the human decision")

## Build it
Complete the `RunLog` (record + read back the stages) and `approval_rate` over a batch.

In [ ]:
class RunLog:
    def __init__(self):
        self.events = []
    def record(self, stage, detail):
        self.events.append((stage, detail))
    def stages(self):
        return [s for s, _ in self.events]

def approval_rate(decisions):
    # the fraction of runs a human approved ("send")
    return sum(1 for d in decisions if d == "send") / len(decisions)

try:
    log = RunLog()
    log.record("trigger", "email 4471")
    log.record("gather", "lookup_order -> shipped")
    log.record("draft", "due Friday")
    log.record("validate", "ok")
    log.record("approve", "send")
    print("trace stages:", log.stages())
    print("approval rate:", approval_rate(["send", "revise", "send", "send"]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `stages()` reads the run back in order &mdash; the same shape as an agent's message trace, which is your #1 debugging surface.
- `approval_rate` turns a pile of runs into **one number** you can track; a falling rate means the drafts are getting worse.
- LangChain's real `BaseCallbackHandler` fires on every tool start/end &mdash; the same events, captured automatically by LangSmith/Langfuse.

## Your turn (open task &mdash; no grader)
Extend `RunLog` to also record the *tool arguments* on each `gather` event, then run it over a couple of
Lab 6 drafts and compute the approval rate you'd assign. **What good looks like:** from the log alone you
can reconstruct exactly what each run did &mdash; and you have a metric that would tell you if quality slipped.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
log = RunLog()
log.record("trigger", "email 4471")
log.record("gather", {"tool": "lookup_order", "args": {"order_id": "4471"}})   # detail carries the tool ARGS
log.record("draft", "due Friday")
log.record("validate", "ok")
log.record("approve", "send")
print("stages    :", log.stages())
print("gather args:", [d for s, d in log.events if s == "gather"])   # auditable from the log alone
print("approval  :", approval_rate(["send", "send", "revise"]))

---
### Key takeaway
Log every run's trigger, tools, draft, validation and decision -- that's how you debug, audit and MEASURE an automation. Once you can measure it (approval rate), you can improve it. Day 5 goes deeper.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>